# Chia CSV lớn thành các chunk

Notebook chia `data/dark.csv` sang `data/chunks/` theo cơ chế ghi file tạm rồi đổi tên, tránh để lại chunk hỏng khi tiến trình bị ngắt.

## 1. Cấu hình

Thao tác chỉ chạy khi đặt `RUN_DATA_SPLIT=1`; mặc định không xóa bất kỳ file nguồn nào.

In [1]:
from pathlib import Path
import os

import pandas as pd

# Thư mục gốc chứa dữ liệu; giống merge.ipynb, tính tương đối từ repo root.
DATA_DIR = Path('data')
# File nguồn để chia nhỏ: dark.csv do merge.ipynb sinh ra (gộp toàn bộ
# dark_*.csv gốc thành một file duy nhất, chưa được chia lại).
SOURCE_FILE = DATA_DIR / 'dark.csv'
# Thư mục đích chứa các chunk dark_1.csv, dark_2.csv, ... mỗi file tối đa
# ROWS_PER_FILE dòng; tách riêng khỏi data/ gốc để không ghi đè lên ba file
# LFS dark.csv/light.csv/animal_chess.csv.
CHUNK_DIR = DATA_DIR / 'chunks'
# Số dòng mỗi file chunk; mặc định 15 triệu dòng/file để mỗi chunk vẫn nạp vừa
# RAM khi huấn luyện (xem train.ipynb đọc lại data/dark_*.csv theo chunk).
ROWS_PER_FILE = int(os.getenv('ROWS_PER_FILE', '15000000'))
# Cổng an toàn: mặc định không ghi gì ra đĩa (chỉ preview cấu hình bên dưới);
# đặt RUN_DATA_SPLIT=1 mới thực sự chạy split_csv.
RUN_SPLIT = os.getenv('RUN_DATA_SPLIT') == '1'
# Cổng phụ, chỉ có tác dụng khi RUN_SPLIT=1: sau khi chia file thành công, nếu
# đặt DELETE_DERIVED_SOURCES=1 thì xoá luôn ba file dẫn xuất lớn (dark.csv,
# light.csv, animal_chess.csv) để tiết kiệm dung lượng, vì dữ liệu đã được
# bảo toàn dưới dạng chunk trong data/chunks/.
DELETE_DERIVED_SOURCES = os.getenv('DELETE_DERIVED_SOURCES') == '1'

# Xem nhanh cấu hình hiệu lực (đường dẫn, file nguồn có tồn tại hay không, số
# dòng/chunk) trước khi quyết định có bật RUN_DATA_SPLIT hay không.
{
    'source': str(SOURCE_FILE),
    'source_exists': SOURCE_FILE.exists(),
    'rows_per_file': ROWS_PER_FILE,
    'output_dir': str(CHUNK_DIR),
}

{'source': 'data\\dark.csv',
 'source_exists': False,
 'rows_per_file': 15000000,
 'output_dir': 'data\\chunks'}

## 2. Hàm chia an toàn

Các file cũ trong `data/chunks/` được thay thế có chủ đích; CSV ở thư mục `data/` không bị đụng tới.

In [2]:
def split_csv(source: Path, output_dir: Path, rows_per_file: int) -> list[Path]:
    if not source.exists():
        raise FileNotFoundError(
            f'{source} chưa tồn tại; hãy chạy merge.ipynb trước.'
        )
    output_dir.mkdir(parents=True, exist_ok=True)
    written = []
    # Đọc theo chunk (rows_per_file dòng/lần) thay vì nạp cả dark.csv (có thể
    # hàng chục triệu dòng) vào RAM một lần; mỗi chunk pandas đọc được ghi
    # thẳng thành một file dark_<index>.csv riêng, đánh số bắt đầu từ 1.
    for index, frame in enumerate(pd.read_csv(source, chunksize=rows_per_file), start=1):
        destination = output_dir / f'dark_{index}.csv'
        # Ghi kiểu atomic: ghi ra file tạm .csv.tmp trước, xong xuôi mới
        # replace() sang tên thật. Nếu notebook bị ngắt giữa chừng (lỗi,
        # Ctrl+C, mất điện) thì destination hoặc chưa tồn tại hoặc còn nguyên
        # nội dung cũ, không bao giờ ở trạng thái file dở dang/hỏng.
        temporary = destination.with_suffix('.csv.tmp')
        frame.to_csv(temporary, index=False)
        temporary.replace(destination)
        written.append(destination)
        print(f'Đã ghi {destination} ({len(frame):,} dòng)')
    return written

In [3]:
if RUN_SPLIT:
    chunks = split_csv(SOURCE_FILE, CHUNK_DIR, ROWS_PER_FILE)
    # Chỉ xoá file dẫn xuất lớn sau khi split_csv trả về thành công (không
    # raise); nếu chia lỗi giữa chừng, file gốc vẫn còn nguyên để chạy lại.
    if DELETE_DERIVED_SOURCES:
        for name in ('dark.csv', 'light.csv', 'animal_chess.csv'):
            (DATA_DIR / name).unlink(missing_ok=True)
        print('Đã xóa các file dẫn xuất lớn sau khi chia thành công.')
    print(f'Hoàn tất {len(chunks)} chunk.')
else:
    # Chế độ mặc định: không đụng tới đĩa, chỉ nhắc cách bật pipeline thật.
    print('Chế độ preview: đặt RUN_DATA_SPLIT=1 để bắt đầu chia file.')

Chế độ preview: đặt RUN_DATA_SPLIT=1 để bắt đầu chia file.


## Tóm tắt

- Chunk được lưu riêng trong `data/chunks/`, không ghi đè ba file LFS gốc.
- Ghi qua `.tmp` rồi đổi tên để hạn chế file dở dang.
- Xóa file dẫn xuất là tùy chọn và chỉ xảy ra sau khi chia thành công.